# 03 — Modeling

This notebook evaluates a global Random Forest baseline and crystal-system-specific Random Forest and Lasso models for predicting DFT-calculated oxide band gaps. All models use leakage-free five-fold GroupKFold validation grouped by reduced formula. Random Forest permutation importance is also evaluated across five random seeds.

## Dataset source

This project-specific dataset was generated from the Materials Project API. I filtered for non-metallic inorganic oxide compounds with valid DFT-calculated band gaps, then added MAGPIE composition descriptors and saved the processed dataset as `zhao_nonmetal_oxide_bandgap_magpie_project.csv`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the repository root
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

data_dir = project_root / "data"

processed_data_path = (
    data_dir
    / "zhao_nonmetal_oxide_bandgap_magpie_project.csv"
)

df = pd.read_csv(processed_data_path)

required_columns = [
    "formula",
    "reduced_formula",
    "band_gap",
    "crystal_system"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

magpie_cols = [
    column
    for column in df.columns
    if "MagpieData" in column
]

print("Processed dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("Unique reduced formulas:", df["reduced_formula"].nunique())
print("Number of MAGPIE features:", len(magpie_cols))

print("\nTarget property: band_gap (eV)")
print(df["band_gap"].describe().round(3))

print("\nCrystal system counts:")
print(df["crystal_system"].value_counts())

Processed dataset loaded successfully.
Dataset shape: (40078, 147)
Unique reduced formulas: 22816
Number of MAGPIE features: 132

Target property: band_gap (eV)
count    40078.000
mean         2.079
std          1.465
min          0.100
25%          0.848
50%          1.806
75%          3.076
max          8.375
Name: band_gap, dtype: float64

Crystal system counts:
crystal_system
Monoclinic      14447
Triclinic        9095
Orthorhombic     8160
Trigonal         2796
Tetragonal       2727
Cubic            1782
Hexagonal        1071
Name: count, dtype: int64


## Global Random Forest Baseline

A single Random Forest model is first evaluated on the complete dataset using MAGPIE descriptors, stability and structure metadata, and one-hot-encoded crystal system. Five-fold GroupKFold cross-validation is grouped by reduced formula to prevent composition leakage.

In [2]:
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# ------------------------------------------------------------
# 1. Define target and features
# ------------------------------------------------------------

target_col = "band_gap"

magpie_cols = [
    c for c in df.columns
    if "MagpieData" in c
]

metadata_features = [
    "nelements",
    "formation_energy_per_atom",
    "energy_above_hull",
    "density",
    "volume",
    "nsites"
]

categorical_features = [
    "crystal_system"
]

feature_cols = (
    magpie_cols
    + metadata_features
    + categorical_features
)

# ------------------------------------------------------------
# 2. Build the modeling dataframe
#    IMPORTANT: group by reduced_formula, not formula
# ------------------------------------------------------------

model_df = df.dropna(
    subset=[
        target_col,
        "reduced_formula"
    ] + feature_cols
).copy()

X = model_df[feature_cols]
y = model_df[target_col]

groups = model_df["reduced_formula"].astype(str)

print("Model dataset shape:", model_df.shape)
print("Number of MAGPIE features:", len(magpie_cols))
print(
    "Total number of features before one-hot encoding:",
    len(feature_cols)
)
print("Group key: reduced_formula")
print(
    "Number of unique reduced-formula groups:",
    groups.nunique()
)

print("\nMost frequent reduced-formula groups:")
print(groups.value_counts().head(10))

# ------------------------------------------------------------
# 3. Preprocessing
# ------------------------------------------------------------

preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            magpie_cols + metadata_features
        )
    ]
)

# ------------------------------------------------------------
# 4. Random Forest baseline
# ------------------------------------------------------------

rf = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

model = Pipeline([
    ("preprocess", preprocess),
    ("rf", rf)
])

# ------------------------------------------------------------
# 5. Leakage-free GroupKFold
# ------------------------------------------------------------

cv = GroupKFold(
    n_splits=5
)

print("\nReduced-formula overlap check:")

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X, y, groups),
    start=1
):
    train_groups = set(
        groups.iloc[train_idx]
    )

    test_groups = set(
        groups.iloc[test_idx]
    )

    overlap = train_groups.intersection(
        test_groups
    )

    print(
        f"Fold {fold}: "
        f"train samples = {len(train_idx):,}, "
        f"test samples = {len(test_idx):,}, "
        f"reduced-formula overlap = {len(overlap)}"
    )

# ------------------------------------------------------------
# 6. Cross-validation
# ------------------------------------------------------------

scores = cross_validate(
    model,
    X,
    y,
    groups=groups,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    },
    n_jobs=1
)

mae_scores = -scores["test_mae"]
r2_scores = scores["test_r2"]

# ------------------------------------------------------------
# 7. Report results
# ------------------------------------------------------------

print("\nBaseline model: Random Forest")
print(
    "Features: MAGPIE descriptors "
    "+ stability/structure metadata"
)
print("CV strategy: GroupKFold, 5 splits")
print("Group key: reduced_formula")

print(
    f"CV MAE = "
    f"{mae_scores.mean():.3f} "
    f"± {mae_scores.std():.3f} eV"
)

print(
    f"CV R² = "
    f"{r2_scores.mean():.3f} "
    f"± {r2_scores.std():.3f}"
)

Model dataset shape: (40078, 147)
Number of MAGPIE features: 132
Total number of features before one-hot encoding: 139
Group key: reduced_formula
Number of unique reduced-formula groups: 22816

Most frequent reduced-formula groups:
reduced_formula
Li9Mn2Co5O16     358
SiO2             314
Li7Mn4CoO12      111
Li7Mn2(CoO4)3    104
Li4MnCo2O7        96
Li6V3P8O29        89
Li3V3P8O29        87
Li4V3P8O29        86
Al2O3             86
LiMnPO4           79
Name: count, dtype: int64

Reduced-formula overlap check:
Fold 1: train samples = 32,062, test samples = 8,016, reduced-formula overlap = 0
Fold 2: train samples = 32,062, test samples = 8,016, reduced-formula overlap = 0
Fold 3: train samples = 32,062, test samples = 8,016, reduced-formula overlap = 0
Fold 4: train samples = 32,063, test samples = 8,015, reduced-formula overlap = 0
Fold 5: train samples = 32,063, test samples = 8,015, reduced-formula overlap = 0

Baseline model: Random Forest
Features: MAGPIE descriptors + stability/st

## Crystal-System-Specific Modeling

Separate Random Forest and Lasso models are evaluated for each crystal system to test whether predictive performance and important descriptors differ across structural groups. Crystal system itself is excluded from these subset models because it is constant within each group.

In [3]:
# Prepare features for crystal-system-specific models

magpie_cols = [
    column
    for column in df.columns
    if "MagpieData" in column
]

metadata_features = [
    "nelements",
    "formation_energy_per_atom",
    "energy_above_hull",
    "density",
    "volume",
    "nsites"
]

# Crystal system is deliberately excluded because it is constant
# inside each crystal-system subset.
system_feature_cols = magpie_cols + metadata_features

model_df = df.dropna(
    subset=[
        "band_gap",
        "reduced_formula",
        "crystal_system"
    ] + system_feature_cols
).copy()

crystal_system_counts = (
    model_df["crystal_system"]
    .value_counts()
    .sort_values(ascending=False)
)

print("Crystal-system sample counts:")
print(crystal_system_counts)

print("\nNumber of features used within each crystal system:")
print(len(system_feature_cols))

print("\nCrystal system is excluded from subset-model features.")

Crystal-system sample counts:
crystal_system
Monoclinic      14447
Triclinic        9095
Orthorhombic     8160
Trigonal         2796
Tetragonal       2727
Cubic            1782
Hexagonal        1071
Name: count, dtype: int64

Number of features used within each crystal system:
138

Crystal system is excluded from subset-model features.


### Random Forest by Crystal System

A separate Random Forest model is evaluated for each crystal system using five-fold GroupKFold cross-validation grouped by reduced formula. The crystal-system label is excluded because it is constant within each subset.

In [4]:
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.ensemble import RandomForestRegressor

crystal_rf_results = []

for crystal_system in crystal_system_counts.index:
    subset = model_df[
        model_df["crystal_system"] == crystal_system
    ].copy()

    X_system = subset[system_feature_cols]
    y_system = subset["band_gap"]
    groups_system = subset["reduced_formula"].astype(str)

    n_unique_groups = groups_system.nunique()

    if n_unique_groups < 5:
        print(
            f"Skipping {crystal_system}: "
            f"only {n_unique_groups} unique reduced-formula groups."
        )
        continue

    rf_system = RandomForestRegressor(
        n_estimators=50,
        random_state=42,
        n_jobs=-1
    )

    cv_system = GroupKFold(n_splits=5)

    scores_system = cross_validate(
        rf_system,
        X_system,
        y_system,
        groups=groups_system,
        cv=cv_system,
        scoring={
            "mae": "neg_mean_absolute_error",
            "r2": "r2"
        },
        n_jobs=1
    )

    mae_system = -scores_system["test_mae"]
    r2_system = scores_system["test_r2"]

    crystal_rf_results.append({
        "crystal_system": crystal_system,
        "n_samples": len(subset),
        "n_reduced_formula_groups": n_unique_groups,
        "mae_mean": mae_system.mean(),
        "mae_std": mae_system.std(),
        "r2_mean": r2_system.mean(),
        "r2_std": r2_system.std()
    })

    print(
        f"{crystal_system}: "
        f"n={len(subset):,}, "
        f"MAE={mae_system.mean():.3f} ± {mae_system.std():.3f} eV, "
        f"R²={r2_system.mean():.3f} ± {r2_system.std():.3f}"
    )

crystal_rf_results_df = pd.DataFrame(
    crystal_rf_results
).sort_values("mae_mean")

results_path = (
    data_dir / "crystal_system_rf_results.csv"
)

crystal_rf_results_df.to_csv(
    results_path,
    index=False
)

print("\nCrystal-system Random Forest results:")
display(crystal_rf_results_df.round(3))

print("\nSaved to:", results_path.resolve())

Monoclinic: n=14,447, MAE=0.567 ± 0.011 eV, R²=0.729 ± 0.010
Triclinic: n=9,095, MAE=0.503 ± 0.024 eV, R²=0.677 ± 0.026
Orthorhombic: n=8,160, MAE=0.606 ± 0.036 eV, R²=0.683 ± 0.012
Trigonal: n=2,796, MAE=0.583 ± 0.014 eV, R²=0.692 ± 0.019
Tetragonal: n=2,727, MAE=0.587 ± 0.030 eV, R²=0.684 ± 0.034
Cubic: n=1,782, MAE=0.572 ± 0.043 eV, R²=0.663 ± 0.033
Hexagonal: n=1,071, MAE=0.667 ± 0.047 eV, R²=0.627 ± 0.079

Crystal-system Random Forest results:


,crystal_system,n_samples,n_reduced_formula_groups,mae_mean,mae_std,r2_mean,r2_std
1,Triclinic,9095,4749,0.503,0.024,0.677,0.026
0,Monoclinic,14447,10061,0.567,0.011,0.729,0.010
5,Cubic,1782,1634,0.572,0.043,0.663,0.033
3,Trigonal,2796,2374,0.583,0.014,0.692,0.019
4,Tetragonal,2727,2333,0.587,0.030,0.684,0.034
2,Orthorhombic,8160,6104,0.606,0.036,0.683,0.012
6,Hexagonal,1071,913,0.667,0.047,0.627,0.079



Saved to: C:\Users\24250\Desktop\zhao-oxide-bandgap-project\data\crystal_system_rf_results.csv


### Crystal-System Random Forest Finding

Random Forest performance differs across crystal systems. The triclinic subset has the lowest cross-validated MAE at approximately 0.503 eV, while the hexagonal subset has the highest MAE at approximately 0.667 eV. The monoclinic model has the highest \(R^2\), approximately 0.729, whereas the hexagonal model has the lowest \(R^2\), approximately 0.627.

The weaker hexagonal performance may partly reflect its smaller sample size and lower chemical diversity, with only 1,071 compounds and 913 reduced-formula groups. These results support analyzing crystal systems separately rather than assuming that one global model performs equally well for all structural groups.

## Lasso Comparison by Crystal System

Lasso regression is evaluated separately within each crystal system using nested five-fold GroupKFold validation grouped by reduced formula. Within every outer training fold, feature values are clipped using quantile bounds learned only from the training data, near-constant features are removed, and the remaining features are standardized. The regularization strength is selected using an inner GroupKFold search with RMSE as the tuning metric. This leakage-free pipeline limits extreme linear extrapolation while preserving an interpretable sparse-model comparison.

In [5]:
# Stable Lasso comparison for every crystal system
# Validation: nested five-fold GroupKFold grouped by reduced formula
# Preprocessing is learned inside each training fold only.

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import Lasso
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
import warnings
from sklearn.exceptions import ConvergenceWarning


# ------------------------------------------------------------
# 1. Training-fold quantile clipping
# ------------------------------------------------------------

class QuantileClipper(BaseEstimator, TransformerMixin):
    """
    Clip every feature using bounds learned only from the training data.

    This limits extreme feature extrapolation while preserving a
    leakage-free pipeline. The validation/test data never determine
    the clipping thresholds.
    """

    def __init__(
        self,
        lower_quantile=0.005,
        upper_quantile=0.995
    ):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    def fit(self, X, y=None):
        X_array = np.asarray(X, dtype=float)

        self.lower_bounds_ = np.nanquantile(
            X_array,
            self.lower_quantile,
            axis=0
        )

        self.upper_bounds_ = np.nanquantile(
            X_array,
            self.upper_quantile,
            axis=0
        )

        return self

    def transform(self, X):
        X_array = np.asarray(X, dtype=float)

        return np.clip(
            X_array,
            self.lower_bounds_,
            self.upper_bounds_
        )


# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

# Regularization strengths tested in the inner GroupKFold search
alpha_grid = np.logspace(
    -3,
    -0.3,
    10
)

lasso_results = []
lasso_fold_results = []
lasso_coefficient_records = []

warnings.filterwarnings(
    "default",
    category=ConvergenceWarning
)


# ------------------------------------------------------------
# 3. Nested GroupKFold for each crystal system
# ------------------------------------------------------------

for crystal_system in crystal_system_counts.index:

    subset = model_df[
        model_df["crystal_system"] == crystal_system
    ].copy()

    X_system = subset[
        system_feature_cols
    ].astype(float)

    y_system = subset[
        "band_gap"
    ].astype(float)

    groups_system = subset[
        "reduced_formula"
    ].astype(str)

    n_unique_groups = groups_system.nunique()

    if n_unique_groups < 5:
        print(
            f"Skipping {crystal_system}: "
            f"only {n_unique_groups} reduced-formula groups."
        )
        continue

    outer_cv = GroupKFold(
        n_splits=5
    )

    fold_mae = []
    fold_rmse = []
    fold_r2 = []
    fold_best_alpha = []

    print(
        f"\nTraining stable Lasso for "
        f"{crystal_system}..."
    )

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(
            X_system,
            y_system,
            groups_system
        ),
        start=1
    ):

        X_train = X_system.iloc[train_idx]
        X_test = X_system.iloc[test_idx]

        y_train = y_system.iloc[train_idx]
        y_test = y_system.iloc[test_idx]

        groups_train = groups_system.iloc[
            train_idx
        ]

        # All preprocessing steps are fitted inside the training fold.
        lasso_pipeline = Pipeline([
            (
                "clipper",
                QuantileClipper(
                    lower_quantile=0.005,
                    upper_quantile=0.995
                )
            ),
            (
                "variance_filter",
                VarianceThreshold(
                    threshold=1e-12
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "lasso",
                Lasso(
                    max_iter=200000,
                    tol=1e-4,
                    selection="cyclic"
                )
            )
        ])

        inner_cv = GroupKFold(
            n_splits=5
        )

        # RMSE is used for selecting alpha because it strongly
        # penalizes catastrophic extrapolation errors.
        search = GridSearchCV(
            estimator=lasso_pipeline,
            param_grid={
                "lasso__alpha": alpha_grid
            },
            scoring="neg_root_mean_squared_error",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            error_score="raise"
        )

        search.fit(
            X_train,
            y_train,
            groups=groups_train
        )

        y_pred = search.predict(
            X_test
        )

        mae = mean_absolute_error(
            y_test,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                y_pred
            )
        )

        r2 = r2_score(
            y_test,
            y_pred
        )

        best_alpha = search.best_params_[
            "lasso__alpha"
        ]

        fold_mae.append(mae)
        fold_rmse.append(rmse)
        fold_r2.append(r2)
        fold_best_alpha.append(best_alpha)

        lasso_fold_results.append({
            "crystal_system": crystal_system,
            "fold": fold,
            "train_n": len(train_idx),
            "test_n": len(test_idx),
            "selected_alpha": best_alpha,
            "mae": mae,
            "rmse": rmse,
            "r2": r2,
            "actual_min": y_test.min(),
            "actual_max": y_test.max(),
            "predicted_min": y_pred.min(),
            "predicted_max": y_pred.max()
        })

        # ----------------------------------------------------
        # Recover standardized Lasso coefficients
        # ----------------------------------------------------

        variance_filter = (
            search.best_estimator_
            .named_steps["variance_filter"]
        )

        retained_mask = (
            variance_filter.get_support()
        )

        retained_coefficients = (
            search.best_estimator_
            .named_steps["lasso"]
            .coef_
        )

        # Use zero for features removed by VarianceThreshold,
        # so all five folds contribute to every feature summary.
        full_coefficients = np.zeros(
            len(system_feature_cols),
            dtype=float
        )

        full_coefficients[
            retained_mask
        ] = retained_coefficients

        for feature, coefficient in zip(
            system_feature_cols,
            full_coefficients
        ):
            lasso_coefficient_records.append({
                "crystal_system": crystal_system,
                "fold": fold,
                "feature": feature,
                "coefficient": coefficient,
                "absolute_coefficient":
                    abs(coefficient),
                "selected_alpha": best_alpha
            })

        print(
            f"  Fold {fold}: "
            f"alpha={best_alpha:.4g}, "
            f"MAE={mae:.3f} eV, "
            f"RMSE={rmse:.3f} eV, "
            f"R²={r2:.3f}, "
            f"prediction range="
            f"[{y_pred.min():.2f}, "
            f"{y_pred.max():.2f}] eV"
        )

    # --------------------------------------------------------
    # Crystal-system summary
    # --------------------------------------------------------

    lasso_results.append({
        "crystal_system": crystal_system,
        "n_samples": len(subset),
        "n_reduced_formula_groups":
            n_unique_groups,
        "mae_mean": np.mean(fold_mae),
        "mae_std": np.std(fold_mae),
        "rmse_mean": np.mean(fold_rmse),
        "rmse_std": np.std(fold_rmse),
        "r2_mean": np.mean(fold_r2),
        "r2_std": np.std(fold_r2),
        "median_selected_alpha":
            np.median(fold_best_alpha)
    })

    print(
        f"{crystal_system} summary: "
        f"MAE={np.mean(fold_mae):.3f} "
        f"± {np.std(fold_mae):.3f} eV, "
        f"RMSE={np.mean(fold_rmse):.3f} "
        f"± {np.std(fold_rmse):.3f} eV, "
        f"R²={np.mean(fold_r2):.3f} "
        f"± {np.std(fold_r2):.3f}"
    )


# ------------------------------------------------------------
# 4. Save model-performance results
# ------------------------------------------------------------

crystal_lasso_results_df = (
    pd.DataFrame(lasso_results)
    .sort_values("mae_mean")
    .reset_index(drop=True)
)

lasso_results_path = (
    data_dir
    / "crystal_system_lasso_results.csv"
)

crystal_lasso_results_df.to_csv(
    lasso_results_path,
    index=False
)


lasso_fold_results_df = pd.DataFrame(
    lasso_fold_results
)

lasso_fold_results_path = (
    data_dir
    / "crystal_system_lasso_fold_results.csv"
)

lasso_fold_results_df.to_csv(
    lasso_fold_results_path,
    index=False
)


# ------------------------------------------------------------
# 5. Summarize and save feature coefficients
# ------------------------------------------------------------

lasso_coefficients_df = pd.DataFrame(
    lasso_coefficient_records
)

lasso_feature_summary_df = (
    lasso_coefficients_df
    .groupby(
        [
            "crystal_system",
            "feature"
        ],
        as_index=False
    )
    .agg(
        mean_coefficient=(
            "coefficient",
            "mean"
        ),
        std_coefficient=(
            "coefficient",
            "std"
        ),
        mean_absolute_coefficient=(
            "absolute_coefficient",
            "mean"
        ),
        std_absolute_coefficient=(
            "absolute_coefficient",
            "std"
        ),
        nonzero_folds=(
            "absolute_coefficient",
            lambda values: int(
                np.sum(
                    np.asarray(values) > 1e-10
                )
            )
        )
    )
)

lasso_feature_path = (
    data_dir
    / "crystal_system_lasso_feature_summary.csv"
)

lasso_feature_summary_df.to_csv(
    lasso_feature_path,
    index=False
)


# ------------------------------------------------------------
# 6. Display final stable results
# ------------------------------------------------------------

print(
    "\nStable crystal-system Lasso results:"
)

display(
    crystal_lasso_results_df.round(3)
)

print("\nSaved summary results to:")
print(lasso_results_path.resolve())

print("\nSaved fold-level results to:")
print(lasso_fold_results_path.resolve())

print("\nSaved feature-coefficient summary to:")
print(lasso_feature_path.resolve())


Training stable Lasso for Monoclinic...
  Fold 1: alpha=0.001, MAE=0.774 eV, RMSE=0.996 eV, R²=0.547, prediction range=[-1.90, 4.95] eV
  Fold 2: alpha=0.001, MAE=0.802 eV, RMSE=1.010 eV, R²=0.547, prediction range=[-2.30, 5.08] eV
  Fold 3: alpha=0.001, MAE=0.816 eV, RMSE=1.025 eV, R²=0.531, prediction range=[-1.96, 4.98] eV
  Fold 4: alpha=0.001, MAE=0.801 eV, RMSE=1.021 eV, R²=0.524, prediction range=[-2.91, 5.38] eV
  Fold 5: alpha=0.001, MAE=0.808 eV, RMSE=1.027 eV, R²=0.544, prediction range=[-3.40, 4.99] eV
Monoclinic summary: MAE=0.800 ± 0.014 eV, RMSE=1.016 ± 0.011 eV, R²=0.538 ± 0.009

Training stable Lasso for Triclinic...
  Fold 1: alpha=0.001, MAE=0.652 eV, RMSE=0.860 eV, R²=0.519, prediction range=[-1.35, 4.55] eV
  Fold 2: alpha=0.001, MAE=0.705 eV, RMSE=0.940 eV, R²=0.456, prediction range=[-2.87, 4.78] eV
  Fold 3: alpha=0.001, MAE=0.630 eV, RMSE=0.834 eV, R²=0.550, prediction range=[-2.66, 4.62] eV
  Fold 4: alpha=0.001, MAE=0.686 eV, RMSE=0.908 eV, R²=0.490, predict

,crystal_system,n_samples,n_reduced_formula_groups,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,median_selected_alpha
0,Triclinic,9095,4749,0.663,0.028,0.884,0.037,0.501,0.032,0.001
1,Trigonal,2796,2374,0.740,0.011,0.954,0.022,0.561,0.029,0.001
2,Cubic,1782,1634,0.770,0.032,0.981,0.040,0.487,0.067,0.002
3,Orthorhombic,8160,6104,0.776,0.007,0.998,0.009,0.539,0.043,0.001
4,Monoclinic,14447,10061,0.800,0.014,1.016,0.011,0.538,0.009,0.001
5,Tetragonal,2727,2333,0.802,0.023,1.010,0.027,0.497,0.077,0.001
6,Hexagonal,1071,913,0.808,0.060,1.060,0.097,0.514,0.089,0.008



Saved summary results to:
C:\Users\24250\Desktop\zhao-oxide-bandgap-project\data\crystal_system_lasso_results.csv

Saved fold-level results to:
C:\Users\24250\Desktop\zhao-oxide-bandgap-project\data\crystal_system_lasso_fold_results.csv

Saved feature-coefficient summary to:
C:\Users\24250\Desktop\zhao-oxide-bandgap-project\data\crystal_system_lasso_feature_summary.csv


### Random Forest versus Lasso

Random Forest outperformed Lasso in every crystal-system group. Random Forest MAE ranged from approximately 0.503 to 0.667 eV, whereas stable Lasso MAE ranged from approximately 0.663 to 0.808 eV. Random Forest also achieved higher \(R^2\) values for all seven crystal systems.

This consistent performance gap suggests that nonlinear relationships and feature interactions are important for predicting oxide band gaps. Random Forest can represent these effects, whereas Lasso is limited to a sparse linear relationship. Lasso nevertheless remains useful as an interpretable comparison model and for identifying influential standardized predictors.

Training-fold quantile clipping was used to limit extreme linear extrapolation without using information from the validation folds. Some Lasso predictions remain negative because ordinary Lasso does not enforce a physical non-negativity constraint. These predictions are retained rather than post-processed and are treated as a limitation of the linear model.

## Permutation Importance across Random Seeds

Permutation importance is calculated on a held-out reduced-formula GroupKFold fold rather than on the training data. For each crystal system, five Random Forest models with different random seeds are trained on the same leakage-free split. Each feature is permuted five times. The reported error bars represent variation across the five model seeds.

In [6]:
# Permutation importance for each crystal system
# Five RF seeds, evaluated on a fixed held-out GroupKFold fold

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold

import numpy as np
import pandas as pd


rf_seeds = [11, 22, 33, 44, 55]

importance_records = []


for crystal_system in crystal_system_counts.index:

    subset = model_df[
        model_df["crystal_system"] == crystal_system
    ].copy()

    X_system = subset[system_feature_cols].astype(float)
    y_system = subset["band_gap"].astype(float)
    groups_system = subset["reduced_formula"].astype(str)

    # Use one fixed leakage-free validation fold
    group_cv = GroupKFold(n_splits=5)

    train_idx, test_idx = next(
        group_cv.split(
            X_system,
            y_system,
            groups_system
        )
    )

    X_train = X_system.iloc[train_idx]
    X_test = X_system.iloc[test_idx]

    y_train = y_system.iloc[train_idx]
    y_test = y_system.iloc[test_idx]

    train_groups = set(groups_system.iloc[train_idx])
    test_groups = set(groups_system.iloc[test_idx])

    overlap = train_groups.intersection(test_groups)

    print(
        f"\n{crystal_system}: "
        f"train={len(train_idx):,}, "
        f"test={len(test_idx):,}, "
        f"reduced-formula overlap={len(overlap)}"
    )

    for seed in rf_seeds:

        rf_importance_model = RandomForestRegressor(
            n_estimators=50,
            random_state=seed,
            n_jobs=-1
        )

        rf_importance_model.fit(
            X_train,
            y_train
        )

        y_pred = rf_importance_model.predict(
            X_test
        )

        test_mae = mean_absolute_error(
            y_test,
            y_pred
        )

        permutation_result = permutation_importance(
            rf_importance_model,
            X_test,
            y_test,
            scoring="neg_mean_absolute_error",
            n_repeats=5,
            random_state=seed,
            n_jobs=-1
        )

        for feature, importance_mean, importance_std in zip(
            system_feature_cols,
            permutation_result.importances_mean,
            permutation_result.importances_std
        ):
            importance_records.append({
                "crystal_system": crystal_system,
                "seed": seed,
                "feature": feature,
                "mae_increase": importance_mean,
                "within_seed_std": importance_std,
                "test_mae": test_mae
            })

        print(
            f"  Seed {seed}: "
            f"held-out MAE={test_mae:.3f} eV"
        )


# Seed-level results
permutation_seed_df = pd.DataFrame(
    importance_records
)

# Rank features independently for every crystal system and seed
permutation_seed_df["rank_within_seed"] = (
    permutation_seed_df
    .groupby(
        ["crystal_system", "seed"]
    )["mae_increase"]
    .rank(
        method="min",
        ascending=False
    )
)


# Summary across the five seeds
permutation_summary_df = (
    permutation_seed_df
    .groupby(
        ["crystal_system", "feature"],
        as_index=False
    )
    .agg(
        mean_mae_increase=(
            "mae_increase",
            "mean"
        ),
        std_across_seeds=(
            "mae_increase",
            "std"
        ),
        mean_within_seed_std=(
            "within_seed_std",
            "mean"
        ),
        top3_seed_count=(
            "rank_within_seed",
            lambda values: int(
                np.sum(
                    np.asarray(values) <= 3
                )
            )
        ),
        positive_seed_count=(
            "mae_increase",
            lambda values: int(
                np.sum(
                    np.asarray(values) > 0
                )
            )
        )
    )
)

permutation_summary_df["mean_rank"] = (
    permutation_seed_df
    .groupby(
        ["crystal_system", "feature"]
    )["rank_within_seed"]
    .mean()
    .values
)

permutation_summary_df = (
    permutation_summary_df
    .sort_values(
        [
            "crystal_system",
            "mean_mae_increase"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# Save both detailed and summarized results
seed_results_path = (
    data_dir
    / "crystal_system_permutation_importance_seeds.csv"
)

summary_results_path = (
    data_dir
    / "crystal_system_permutation_importance_summary.csv"
)

permutation_seed_df.to_csv(
    seed_results_path,
    index=False
)

permutation_summary_df.to_csv(
    summary_results_path,
    index=False
)


print("\nTop five permutation features by crystal system:")

for crystal_system in crystal_system_counts.index:

    top_features = (
        permutation_summary_df[
            permutation_summary_df[
                "crystal_system"
            ] == crystal_system
        ]
        .nlargest(
            5,
            "mean_mae_increase"
        )
        [
            [
                "feature",
                "mean_mae_increase",
                "std_across_seeds",
                "top3_seed_count"
            ]
        ]
    )

    print(f"\n{crystal_system}:")
    display(
        top_features.round(4)
    )


print("\nSaved seed-level results to:")
print(seed_results_path.resolve())

print("\nSaved summary results to:")
print(summary_results_path.resolve())


Monoclinic: train=11,557, test=2,890, reduced-formula overlap=0
  Seed 11: held-out MAE=0.586 eV
  Seed 22: held-out MAE=0.575 eV
  Seed 33: held-out MAE=0.582 eV
  Seed 44: held-out MAE=0.584 eV
  Seed 55: held-out MAE=0.580 eV

Triclinic: train=7,276, test=1,819, reduced-formula overlap=0
  Seed 11: held-out MAE=0.516 eV
  Seed 22: held-out MAE=0.516 eV
  Seed 33: held-out MAE=0.506 eV
  Seed 44: held-out MAE=0.513 eV
  Seed 55: held-out MAE=0.510 eV

Orthorhombic: train=6,528, test=1,632, reduced-formula overlap=0
  Seed 11: held-out MAE=0.655 eV
  Seed 22: held-out MAE=0.657 eV
  Seed 33: held-out MAE=0.685 eV
  Seed 44: held-out MAE=0.688 eV
  Seed 55: held-out MAE=0.645 eV

Trigonal: train=2,236, test=560, reduced-formula overlap=0
  Seed 11: held-out MAE=0.574 eV
  Seed 22: held-out MAE=0.597 eV
  Seed 33: held-out MAE=0.594 eV
  Seed 44: held-out MAE=0.581 eV
  Seed 55: held-out MAE=0.584 eV

Tetragonal: train=2,181, test=546, reduced-formula overlap=0
  Seed 11: held-out MAE=

,feature,mean_mae_increase,std_across_seeds,top3_seed_count
276,energy_above_hull,0.2897,0.0021,5
277,formation_energy_per_atom,0.1611,0.0012,5
278,MagpieData maximum NdValence,0.1203,0.0214,3
279,MagpieData range NdValence,0.0981,0.0205,2
280,MagpieData maximum NValence,0.0525,0.0055,0



Triclinic:


,feature,mean_mae_increase,std_across_seeds,top3_seed_count
690,energy_above_hull,0.1509,0.0032,5
691,formation_energy_per_atom,0.1093,0.0070,5
692,MagpieData avg_dev GSvolume_pa,0.0567,0.0071,4
693,MagpieData avg_dev NdValence,0.0460,0.0094,1
694,MagpieData mean NdValence,0.0382,0.0111,0



Orthorhombic:


...


Cubic:


,feature,mean_mae_increase,std_across_seeds,top3_seed_count
0,formation_energy_per_atom,0.2826,0.0242,5
1,energy_above_hull,0.2161,0.0203,5
2,MagpieData avg_dev NdValence,0.1191,0.0195,4
3,MagpieData mean NdValence,0.0702,0.0250,1
4,nsites,0.0303,0.0094,0



Hexagonal:


,feature,mean_mae_increase,std_across_seeds,top3_seed_count
138,energy_above_hull,0.2840,0.0277,5
139,MagpieData mean NdValence,0.1577,0.0396,4
140,formation_energy_per_atom,0.1447,0.0163,5
141,MagpieData avg_dev NdValence,0.1059,0.0199,1
142,MagpieData mean MendeleevNumber,0.0490,0.0078,0



Saved seed-level results to:
C:\Users\24250\Desktop\zhao-oxide-bandgap-project\data\crystal_system_permutation_importance_seeds.csv

Saved summary results to:
C:\Users\24250\Desktop\zhao-oxide-bandgap-project\data\crystal_system_permutation_importance_summary.csv
